<a href="https://colab.research.google.com/github/NickLarsonUVA/DS4002_CS3_BitcoinVSInflation/blob/main/CS3_Time_series_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS 4002 CS3: Bitcoin vs. Inflation - Time Series Analysis
This notebook provides the complete analytical pipeline to evaluate whether Bitcoin acts as a macroeconomic hedge against inflation. It utilizes a 4-variable Vector Autoregression (VAR) model, controlling for broader equity market movements via the S&P 500.

The workflow is divided into four main sections: Data Aggregation & Log Returns, Stationarity Testing, VAR Modeling & Stability, and Impulse Response Analysis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import warnings

# Suppress minor warnings for clean output
warnings.filterwarnings("ignore")

# Set visualization style
sns.set_theme(style="whitegrid", palette="muted")

# Create an output directory for visuals and tables
output_folder = "OUTPUT"
os.makedirs(output_folder, exist_ok=True)

## Part 1: Data Aggregation & Log Returns
The following section loads the daily master dataset. The data is resampled into monthly frequencies. To ensure the financial time series are modeled appropriately, percentage changes are calculated using continuous logarithmic returns rather than simple arithmetic returns.

In [ ]:
# Load the daily dataset
input_file = "master_dataset_daily.csv"
df_daily = pd.read_csv(input_file)
df_daily["date"] = pd.to_datetime(df_daily["date"], errors="coerce")
df_daily = df_daily.dropna(subset=["date"])

# Set date as index for proper time-series resampling
df_daily = df_daily.set_index("date")

# Aggregate to monthly data and calculate log returns
# 1. Bitcoin (Average price for the month, then LOG return)
btc_monthly = df_daily.resample("MS")["btc_price"].mean().reset_index()
btc_monthly.rename(columns={"date": "month"}, inplace=True)
btc_monthly["btc_return"] = np.log(btc_monthly["btc_price"] / btc_monthly["btc_price"].shift(1)) * 100

# 2. S&P 500 (Last close price of the month, then LOG return)
sp500_monthly = df_daily.resample("MS")["sp500_close"].last().reset_index()
sp500_monthly.rename(columns={"date": "month"}, inplace=True)
sp500_monthly["sp500_return"] = np.log(sp500_monthly["sp500_close"] / sp500_monthly["sp500_close"].shift(1)) * 100

# 3. 5-Year Breakeven Inflation (Average expectation for the month)
t5y_monthly = df_daily.resample("MS")["breakeven_5y"].mean().reset_index()
t5y_monthly.rename(columns={"date": "month"}, inplace=True)

# 4. CPI YoY Inflation (Last reported rate for the month)
cpi_monthly = df_daily.resample("MS")["cpi_yoy_pct"].last().reset_index()
cpi_monthly.rename(columns={"date": "month"}, inplace=True)

# Merge all variables together
df = btc_monthly.merge(sp500_monthly, on="month", how="inner")
df = df.merge(t5y_monthly, on="month", how="inner")
df = df.merge(cpi_monthly, on="month", how="inner")

print("Data successfully aggregated and merged.")
display(df.head())

## Part 2: Stationarity Testing & Differencing
Vector Autoregression models require all inputs to be stationary. The Augmented Dickey-Fuller (ADF) test is applied to check for unit roots. Non-stationary variables (typically inflation metrics) are subjected to first-order differencing to achieve stationarity.

In [ ]:
# First differencing for non-stationary inflation variables
df["cpi_yoy_diff"] = df["cpi_yoy_pct"].diff()
df["breakeven_5y_diff"] = df["breakeven_5y"].diff()

# Drop rows with NaN values created by shift() and diff()
df_stat = df.dropna().reset_index(drop=True)

def run_adf_test(series, series_name):
    result = adfuller(series)
    print(f"--- ADF Test: {series_name} --- | p-value: {result[1]:.4f}", end=" | ")
    if result[1] <= 0.05:
        print("STATIONARY")
    else:
        print("NON-STATIONARY")

print("--- Stationarity Check ---")
run_adf_test(df_stat["btc_return"], "Bitcoin Log Returns")
run_adf_test(df_stat["sp500_return"], "S&P 500 Log Returns")
run_adf_test(df_stat["cpi_yoy_diff"], "Differenced CPI YoY")
run_adf_test(df_stat["breakeven_5y_diff"], "Differenced 5-Year Breakeven")

## Part 3: VAR Modeling & Stability Verification
The optimal lag length for the model is selected based on information criteria (AIC and BIC). A parsimonious approach is favored to avoid overfitting. Following model estimation, an eigenvalue stability check is performed to ensure the reliability of the resulting impulse responses.

In [ ]:
print("--- Estimating 4-Variable VAR Model ---")
var_df = df_stat[["btc_return", "sp500_return", "cpi_yoy_diff", "breakeven_5y_diff"]].copy()

model = VAR(var_df)
lag_results = model.select_order(maxlags=6)

# Extract AIC and BIC lags
aic_lag = int(lag_results.aic) if lag_results.aic is not None else 1
bic_lag = int(lag_results.bic) if lag_results.bic is not None else 1

# Choose the more parsimonious (smaller) specification
selected_lag = min(max(1, aic_lag), max(1, bic_lag))
print(f"AIC suggested: {aic_lag} lags")
print(f"BIC suggested: {bic_lag} lags")
print(f"Selected parsimonious lag length: {selected_lag}")

var_model = model.fit(selected_lag)

print("\n--- VAR Stability Check ---")
is_stable = var_model.is_stable(verbose=True)
if is_stable:
    print("Conclusion: The VAR model meets stability requirements. Results are reliable.")
else:
    print("Conclusion: The VAR model is UNSTABLE. Results may be unreliable.")

display(var_model.summary())

## Part 4: Impulse Response Functions (IRFs)
Impulse Response Functions trace the dynamic effect of a one-standard-deviation shock to one variable on the current and future values of the other variables. These visualizations are the primary mechanism for concluding whether Bitcoin responds favorably to inflationary shocks.

In [ ]:
print("Generating Impulse Response plots...")
irf = var_model.irf(10)

# Full 4x4 Matrix
fig1 = irf.plot(orth=False)
plt.suptitle("VAR Impulse Response Functions (incl. S&P 500)", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

# Specific Plot: S&P 500 Shock -> Bitcoin Return
fig2 = irf.plot(impulse='sp500_return', response='btc_return', orth=False)
plt.title("Response of Bitcoin to an S&P 500 Shock", fontsize=14)
plt.tight_layout()
plt.show()

# Specific Plot: CPI Shock -> Bitcoin Return
fig3 = irf.plot(impulse='cpi_yoy_diff', response='btc_return', orth=False)
plt.title("Response of Bitcoin to a CPI Inflation Shock", fontsize=14)
plt.tight_layout()
plt.show()

## Spoilers ahead. Do not read this until you've completed the code scripts above and have reasoned out your results. Use information below to confirm your personal findings.

### Analysis results:
 The final VAR model rejects the "Bitcoin as an inflation hedge" hypothesis. Analysis of the Impulse Response Functions (IRFs) reveals that a shock to CPI inflation (unexpectedly rising prices) actually leads to an immediate negative impact on Bitcoin returns. This suggests that when inflation surprises the market, investors do not flock to Bitcoin as "digital gold." Instead, it is likely that tighter monetary policy and higher interest rates are anticipated, causing a sell-off in speculative assets. Conversely, the S&P 500 shock demonstrates a positive correlation with Bitcoin, confirming that the cryptocurrency behaves more like a high-risk tech equity than a traditional safe-haven asset.

To ensure the results were mathematically sound, critical "data hygiene" steps were performed. First, the Augmented Dickey-Fuller (ADF) test was utilized to check for stationarity. Raw inflation data often exhibits a "trend" (moving up or down over years), which can trick a model into finding false correlations. By "differencing" the data (measuring the change month-to-month rather than the raw level), the data was made stationary, ensuring the results reflect real relationships rather than coincidental trends. Information Criteria (AIC and BIC) were then used to select the optimal "lag" length. This allowed the model to look back exactly one month to find patterns without becoming over-complicated or "noisy," which can occur if a model looks too far into the past.

The inclusion of the S&P 500 as a control variable was perhaps the most critical update to the methodology. Without it, one could argue that Bitcoin was dropping because the broader economy was in a downturn, not specifically because of inflation. By including the S&P 500, Bitcoin's specific reaction to inflation was isolated. Even after accounting for general stock market movements, the negative relationship between inflation shocks and Bitcoin remained clear. This adds significant academic weight to the project, proving that Bitcoin’s volatility in the face of inflation is a unique characteristic of the asset itself, not merely a byproduct of general market movements.

##TLDR:

Result: Bitcoin is NOT a hedge against inflation. When inflation rises unexpectedly, Bitcoin prices tend to drop.

The "Digital Gold" Myth: The model indicates Bitcoin acts more like a risky equity (correlated with the S&P 500) than a safe-haven asset like gold.

Mathematical Rigor: ADF tests were used to make the data "stationary" so results were not a fluke of long-term trends. AIC/BIC criteria ensured the model avoided "overfitting" (finding patterns that do not exist).

S&P 500 Control: The stock market was included in the model to prove that Bitcoin's reaction to inflation is an independent phenomenon, rather than Bitcoin simply following the broader market's lead.